# Using omol and ASE

## Ground-state opt

In [2]:
import torch, os, subprocess, sys
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH has cuda?:", "cuda" in (os.environ.get("LD_LIBRARY_PATH","").lower()))

torch: 2.6.0+cu124 cuda available: True
CUDA_VISIBLE_DEVICES: MIG-250bc972-7bb0-5cb3-a6e6-e54a0fa3227f
LD_LIBRARY_PATH has cuda?: True


In [1]:
# NEW: base paths
LAB_BASE = "/n/jacobsen_lab/Everyone/msak"
FAIRCHEM_CACHE = f"{LAB_BASE}/fairchem_cache"
HF_CACHE = f"{LAB_BASE}/hf_cache"

import os
os.makedirs(FAIRCHEM_CACHE, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)

# Route Hugging Face caches to lab storage (belt & suspenders)
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_HOME"] = f"{LAB_BASE}/.hf"  # optional, also in the priority chain

In [9]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Literal
import time, os, math, shutil
from functools import lru_cache

import numpy as np
from ase import Atoms
from ase.io import read, write
from ase.optimize import LBFGS, BFGS, BFGSLineSearch, FIRE, QuasiNewton
from fairchem.core import FAIRChemCalculator, pretrained_mlip

# -------------------------
# Cache locations & speedups
# -------------------------
# You asked to keep downloads under lab storage by default
LAB_BASE = os.environ.get("UMA_CACHE_BASE", "/n/jacobsen_lab/Everyone/msak")
FAIRCHEM_CACHE = os.path.join(LAB_BASE, "fairchem_cache")  # durable cache
LOCAL_CACHE = os.path.join(os.environ.get("TMPDIR", "/scratch"), "fairchem_cache")  # fast node-local

os.makedirs(FAIRCHEM_CACHE, exist_ok=True)
# Optional: make HF hub fully offline if weights exist locally (avoids network checks)
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

def _stage_cache_to_local(src: str, dst: str) -> str:
    """
    Prefer node-local SSD/NVMe for speed.
    First run on a node copies from lab cache; subsequent runs are instant.
    Falls back to 'src' if copy fails.
    """
    try:
        if os.path.isdir(src):
            if not os.path.isdir(dst):
                os.makedirs(dst, exist_ok=True)
                shutil.copytree(src, dst, dirs_exist_ok=True)
        return dst if (os.path.isdir(dst) and os.listdir(dst)) else src
    except Exception:
        return src

@lru_cache(maxsize=None)
def _get_predict_cached(model: str, device: Optional[str], cache_dir_key: str):
    """
    Single in-memory UMA instance per (model, device, cache_dir). Subsequent calls are instant.
    """
    cache_dir = cache_dir_key or None
    t0 = time.time()
    # Be tolerant of older fairchem versions that may not accept cache_dir
    try:
        pred = pretrained_mlip.get_predict_unit(model, device=device, cache_dir=cache_dir)
    except TypeError:
        pred = pretrained_mlip.get_predict_unit(model, device=device)
    dt = time.time() - t0
    print(f"[UMA init] model={model} device={device or 'auto'} cache={cache_dir or '<default>'}  took {dt:.2f}s")
    return pred

def _resolve_cache_dir(cache_dir: Optional[str], use_local_scratch: bool) -> str:
    base = cache_dir or FAIRCHEM_CACHE
    return _stage_cache_to_local(base, LOCAL_CACHE) if use_local_scratch else base

def build_calculator(model: str = "uma-m-1p1",
                     device: Optional[str] = None,
                     cache_dir: Optional[str] = None,
                     use_local_scratch: bool = True) -> FAIRChemCalculator:
    """
    UMA model + OMol task as an ASE calculator, with memoized predictor & optional local staging.
    """
    resolved = _resolve_cache_dir(cache_dir, use_local_scratch)
    key = os.path.abspath(resolved) if resolved else ""
    predictor = _get_predict_cached(model, device, key)
    return FAIRChemCalculator(predictor, task_name="omol")

# -------------------------
# Constants & unit helpers
# -------------------------
HARTREE_PER_EV = 1.0/27.211386245988
BOHR_PER_ANG  = 1.0/0.529177210903
# convert eV/Å -> Hartree/Bohr
EV_A_to_HB = HARTREE_PER_EV / BOHR_PER_ANG  # ~0.019446905

# Gaussian default convergence baselines (au): G16/G09 conventions
BASE = dict(
    GRMS = 3.0e-4,  # Hartree/Bohr
    GMAX = 4.5e-4,  # Hartree/Bohr
    DRMS = 1.2e-3,  # Bohr
    DMAX = 1.8e-3,  # Bohr
)

# Gaussian nomenclature → target RMS force (others scale proportionally from BASE)
GAUSS_RMS_FORCE = {
    "Loose":      1.7e-3,  # Hartree/Bohr (Opt=Loose)
    "Normal":     3.0e-4,  # default
    "Tight":      1.0e-5,
    "VeryTight":  1.0e-6,
}

OptimName = Literal["LBFGS","BFGS","BFGSLineSearch","FIRE","QuasiNewton"]
OPTIMIZERS = {
    "LBFGS": LBFGS,
    "BFGS": BFGS,
    "BFGSLineSearch": BFGSLineSearch,
    "FIRE": FIRE,
    "QuasiNewton": QuasiNewton,
}

@dataclass
class GaussCutoffs:
    grms: float
    gmax: float
    drms: float
    dmax: float

def gaussian_cutoffs(
    mode: Literal["Loose","Normal","Tight","VeryTight"]="Tight"
) -> GaussCutoffs:
    """
    Map Gaussian-style settings to 4 convergence thresholds.
    - 'mode' picks a target RMS force in Hartree/Bohr and scales others.
    """
    target_grms = GAUSS_RMS_FORCE[mode]
    scale = target_grms / BASE["GRMS"]
    return GaussCutoffs(
        grms = BASE["GRMS"] * scale,
        gmax = BASE["GMAX"] * scale,
        drms = BASE["DRMS"] * scale,
        dmax = BASE["DMAX"] * scale,
    )

def load_xyz(path: str) -> Atoms:
    atoms = read(path)           # ASE will infer format from .xyz
    atoms.pbc = False
    return atoms

def set_charge_mult(atoms: Atoms, charge: int=0, multiplicity: int=1):
    # OMol expects these in atoms.info
    atoms.info.update({"charge": charge, "spin": multiplicity})

def force_metrics_HB(forces_evA: np.ndarray) -> Tuple[float,float]:
    mags = np.linalg.norm(forces_evA, axis=1) * EV_A_to_HB
    grms = float(np.sqrt((mags**2).mean()))
    gmax = float(mags.max())
    return grms, gmax

def disp_metrics_bohr(prev_pos_A: np.ndarray, curr_pos_A: np.ndarray) -> Tuple[float,float]:
    if prev_pos_A is None:
        return 0.0, 0.0
    disp = curr_pos_A - prev_pos_A
    mags = np.linalg.norm(disp, axis=1) * BOHR_PER_ANG
    drms = float(np.sqrt((mags**2).mean()))
    dmax = float(mags.max())
    return drms, dmax

def print_gaussian_table(step:int, grms, gmax, drms, dmax, cuts: GaussCutoffs, dE):
    print(f"\n Step {step:4d}")
    print("         Item               Value     Threshold  Converged?")
    print(f" Maximum Force         {gmax:12.6f}   {cuts.gmax:10.6f}     {'YES' if gmax < cuts.gmax else 'NO'}")
    print(f" RMS     Force         {grms:12.6f}   {cuts.grms:10.6f}     {'YES' if grms < cuts.grms else 'NO'}")
    print(f" Maximum Displacement  {dmax:12.6f}   {cuts.dmax:10.6f}     {'YES' if dmax < cuts.dmax else 'NO'}")
    print(f" RMS     Displacement  {drms:12.6f}   {cuts.drms:10.6f}     {'YES' if drms < cuts.drms else 'NO'}")
    print(f" ΔE(this step) = {dE:+.8e} Hartree  (values in Hartree/Bohr and Bohr)")

def gaussian_converged(grms, gmax, drms, dmax, cuts: GaussCutoffs) -> bool:
    return (grms < cuts.grms) and (gmax < cuts.gmax) and (drms < cuts.drms) and (dmax < cuts.dmax)

def optimize_xyz(
    xyz_path: str,
    *,
    charge: int=0,
    multiplicity: int=1,
    model: str="uma-m-1p1",
    device: Optional[str]='cuda',

    # Gaussian-like knobs:
    mode: Literal["Loose","Normal","Tight","VeryTight"]="Normal",
    maxcycles: int=300,
    optimizer: OptimName="LBFGS",
    maxstep: Optional[float]=None,
    damp: Optional[float]=None,      # FIRE only
    traj: Optional[str]="opt.traj",
    logfile: Optional[str]="opt_ase.log",
    write_final_xyz: Optional[str]="opt_final.xyz",

    # Caching knobs:
    cache_dir: Optional[str]=None,   # default -> FAIRCHEM_CACHE
    use_local_scratch: bool=True,    # stage cache to /scratch for speed
):
    """
    XYZ-in geometry optimization with OMol+ASE using Gaussian-style convergence.
    - Uses a memoized UMA predictor (fast after first call).
    - Optionally stages cache to node-local scratch for faster weight loads.
    Writes:
      - ASE log (opt_ase.log), trajectory (opt.traj), final XYZ (opt_final.xyz)
      - Prints a Gaussian-like 4-criterion table each step in the notebook output.
    Returns (result dict).
    """
    atoms = load_xyz(xyz_path)
    set_charge_mult(atoms, charge, multiplicity)

    atoms.calc = build_calculator(
        model=model, device=device,
        cache_dir=cache_dir,
        use_local_scratch=use_local_scratch,
    )

    Opt = OPTIMIZERS[optimizer]
    opt_kwargs: Dict = {}
    if logfile is not None:
        opt_kwargs["logfile"] = logfile
    if maxstep is not None:
        opt_kwargs["maxstep"] = maxstep
    if optimizer == "FIRE" and damp is not None:
        opt_kwargs["damp"] = damp

    dyn = Opt(atoms, trajectory=traj, **opt_kwargs)

    # we control convergence ourselves → set ASE fmax very small
    ase_fmax = 1e-12  # eV/Å equivalent; ensures ASE doesn't stop before our criteria

    cuts = gaussian_cutoffs(mode=mode)

    prev_pos = None
    prev_E = atoms.get_potential_energy() * HARTREE_PER_EV  # Hartree
    t0 = time.time()
    steps = 0
    converged = False

    for _ase_converged in dyn.irun(fmax=ase_fmax, steps=maxcycles):
        steps = dyn.get_number_of_steps()

        forces = atoms.get_forces()  # eV/Å
        grms, gmax = force_metrics_HB(forces)
        curr_pos = atoms.get_positions()  # Å
        drms, dmax = disp_metrics_bohr(prev_pos, curr_pos)
        E = atoms.get_potential_energy() * HARTREE_PER_EV  # Hartree
        dE = E - prev_E

        print_gaussian_table(steps, grms, gmax, drms, dmax, cuts, dE)

        if gaussian_converged(grms, gmax, drms, dmax, cuts):
            converged = True
            break

        prev_pos = curr_pos.copy()
        prev_E = E

    wall = time.time() - t0
    if write_final_xyz:
        write(write_final_xyz, atoms)

    res = dict(
        converged=converged,
        steps=steps,
        energy_H=float(E),
        fmax_HB=float(gmax),
        traj=traj,
        ase_log=logfile,
        final_xyz=write_final_xyz,
        walltime_s=wall,
        cutoffs=cuts.__dict__,
        mode=mode,
        optimizer=optimizer,
        model=model,
        device=device or "auto",
    )
    print("\nConverged!" if converged else "\nStopped (maxcycles reached).")
    return res


In [12]:
res = optimize_xyz(
    "/n/home10/msak/umadriver/h2o.xyz",
    charge=0, multiplicity=1,
    mode="Tight",            # Gaussian: Loose/Normal/Tight/VeryTight
    optimizer="LBFGS",       # default
    maxcycles=400,
    maxstep=0.15,            # Å (good general cap)
    model="uma-m-1p1",       # default is UMA-M-1p1
    device='cuda',             # let FAIRChem pick; set "cuda" to force GPU
)

res

: 